### Dependencies

In [1]:
# Cell 0 — ONLY if the runtime has just started
!pip install -q huggingface_hub==0.23.0 transformers==4.39.3 diffusers==0.27.2 peft==0.10.0
!pip install -q insightface onnxruntime-gpu onnx onnx2torch
!pip install -q git+https://github.com/tencent-ailab/IP-Adapter.git

print("✓ Libraries installed")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 134.8/134.8 kB 5.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 401.2/401.2 kB 19.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.8/8.8 MB 82.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 41.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 199.1/199.1 kB 13.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 71.0 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
datasets 4.0.0 requires huggingface-hub>=0.24.0, but you have huggingface-hub 0.23.0 which is incompatible.
sentence-transformers 5.4.1 requires transformers<6.0.0,>=4.41.0, but you have transformers 4.39.3 which is incompatible.
gradio 5.50.0 requires huggingface-hub<2.0,>=0.33.5, but you have huggingface-hub 0.23.0 which is incompatible.
     ━━━━

### Drive

In [2]:
# ============================================================
# Cell 1 — Drive, paths and src
# ============================================================

from google.colab import drive
drive.mount("/content/drive")

import os, sys, shutil, importlib, torch
from collections import Counter

DRIVE = "/content/drive/MyDrive/Notebook4"

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
DTYPE = torch.float16

print("DEVICE:", DEVICE)
print("DTYPE:", DTYPE)
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NO GPU")
print("DRIVE:", DRIVE)

# Recargar src desde Drive
if os.path.exists("/content/src"):
    shutil.rmtree("/content/src")

shutil.copytree(f"{DRIVE}/src", "/content/src")

if "/content" not in sys.path:
    sys.path.insert(0, "/content")

for m in list(sys.modules.keys()):
    if m.startswith("src"):
        del sys.modules[m]

importlib.invalidate_caches()

from src.bap2 import BAP2
from src.att2 import attach_bap2

print("✓ src recargado")
print("BAP2:", BAP2)
print("attach_bap2:", attach_bap2)

Mounted at /content/drive
DEVICE: cuda
DTYPE: torch.float16
GPU: Tesla T4
DRIVE: /content/drive/MyDrive/Notebook4
✓ src recargado
BAP2: <class 'src.bap2.BAP2'>
attach_bap2: <function attach_bap2 at 0x7c12310814e0>


## Models

### Stable Diffusion

In [3]:
# Cell 3 - StableDiffusionPipeline

from diffusers import StableDiffusionPipeline, DDIMScheduler, AutoencoderKL
from huggingface_hub import hf_hub_download
from ip_adapter.ip_adapter_faceid import IPAdapterFaceID

DEVICE = "cuda"
DTYPE = torch.float16

vae = AutoencoderKL.from_pretrained(
    "stabilityai/sd-vae-ft-mse",
    torch_dtype=DTYPE
).to(DEVICE)

pipe = StableDiffusionPipeline.from_pretrained(
    "SG161222/Realistic_Vision_V4.0_noVAE",
    vae=vae,
    torch_dtype=DTYPE,
    safety_checker=None
).to(DEVICE)

pipe.scheduler = DDIMScheduler.from_config(pipe.scheduler.config)

ip_ckpt = hf_hub_download(
    repo_id="h94/IP-Adapter-FaceID",
    filename="ip-adapter-faceid_sd15.bin"
)

print("✓ pipeline limpio")
print(type(pipe.unet))

The cache for model files in Transformers v4.22.0 has been updated. Migrating your old cache. This is a one-time only operation. You can interrupt this and resume the migration later on by calling `transformers.utils.move_cache()`.


0it [00:00, ?it/s]

/usr/local/lib/python3.12/dist-packages/huggingface_hub/file_download.py:1132: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_token.py:89: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/547 [00:00<?, ?B/s]

diffusion_pytorch_model.safetensors:   0%|          | 0.00/335M [00:00<?, ?B/s]

...

scheduler_config.json:   0%|          | 0.00/548 [00:00<?, ?B/s]

diffusion_pytorch_model.safetensors:   0%|          | 0.00/3.44G [00:00<?, ?B/s]

Loading pipeline components...:   0%|          | 0/5 [00:00<?, ?it/s]

ip-adapter-faceid_sd15.bin:   0%|          | 0.00/96.7M [00:00<?, ?B/s]

✓ pipeline limpio
<class 'diffusers.models.unets.unet_2d_condition.UNet2DConditionModel'>


### ArcFace

In [4]:
# Cell 4 — ArcFace PyTorch
import onnx, onnx2torch
from insightface.app import FaceAnalysis

# Initialize insightface so it downloads buffalo_l
app = FaceAnalysis(name="buffalo_l", providers=["CUDAExecutionProvider"])
app.prepare(ctx_id=0, det_size=(320, 320))

# Convertir ONNX → PyTorch (gotcha 13.14: .float() obligatorio)
ONNX_PATH = "/root/.insightface/models/buffalo_l/w600k_r50.onnx"
onnx_model = onnx.load(ONNX_PATH)
arcface_torch = onnx2torch.convert(onnx_model).to(DEVICE).eval()
arcface_torch = arcface_torch.float()  # buffers + params a fp32

for p in arcface_torch.parameters():
    p.requires_grad = False

print("✓ ArcFace loaded and converted to PyTorch")
print(f"  Parameters: {sum(p.numel() for p in arcface_torch.parameters()):,}")
print(f"  requires_grad on any param: {any(p.requires_grad for p in arcface_torch.parameters())}")
print(f"  VRAM usada: {torch.cuda.memory_allocated()/1e9:.2f} GB")

download_path: /root/.insightface/models/buffalo_l


100%|██████████| 281857/281857 [00:03<00:00, 71115.42KB/s]


Applied providers: ['CUDAExecutionProvider', 'CPUExecutionProvider'], with options: {'CPUExecutionProvider': {}, 'CUDAExecutionProvider': {'sdpa_kernel': '0', 'use_tf32': '1', 'fuse_conv_bias': '0', 'prefer_nhwc': '0', 'tunable_op_max_tuning_duration_ms': '0', 'enable_skip_layer_norm_strict_mode': '0', 'tunable_op_tuning_enable': '0', 'tunable_op_enable': '0', 'use_ep_level_unified_stream': '0', 'device_id': '0', 'has_user_compute_stream': '0', 'gpu_external_empty_cache': '0', 'cudnn_conv_algo_search': 'EXHAUSTIVE', 'cudnn_conv1d_pad_to_nc1d': '0', 'gpu_mem_limit': '18446744073709551615', 'gpu_external_alloc': '0', 'gpu_external_free': '0', 'arena_extend_strategy': 'kNextPowerOfTwo', 'do_copy_in_default_stream': '1', 'enable_cuda_graph': '0', 'user_compute_stream': '0', 'cudnn_conv_use_max_workspace': '1'}}
find model: /root/.insightface/models/buffalo_l/1k3d68.onnx landmark_3d_68 ['None', 3, 192, 192] 0.0 1.0
Applied providers: ['CUDAExecutionProvider', 'CPUExecutionProvider'], with o

### IP-Adapter

In [5]:
# Cell 5 - IPAdapterFaceID

ip_model = IPAdapterFaceID(
    pipe,
    ip_ckpt,
    device=DEVICE,
    num_tokens=4
)

ip_model.image_proj_model = ip_model.image_proj_model.to(dtype=DTYPE)

print("✓ Official IP-Adapter loaded")
print("processors:", len(pipe.unet.attn_processors))
print(type(pipe.unet))

from collections import Counter

types = Counter(type(proc).__name__ for proc in pipe.unet.attn_processors.values())

print("\nProcessors after IP-Adapter:")
for k, v in types.items():
    print(k, ":", v)

n_ip = 0
for name, proc in pipe.unet.attn_processors.items():
    if hasattr(proc, "to_k_ip") and hasattr(proc, "to_v_ip"):
        n_ip += 1

print("\nProcessors with to_k_ip/to_v_ip:", n_ip)
assert n_ip == 16, "IP-Adapter was not installed correctly: expected 16 processors with to_k_ip/to_v_ip"

/usr/local/lib/python3.12/dist-packages/diffusers/models/lora.py:208: FutureWarning: `LoRALinearLayer` is deprecated and will be removed in version 1.0.0. Use of `LoRALinearLayer` is deprecated. Please switch to PEFT backend by installing PEFT: `pip install peft`.
  deprecate("LoRALinearLayer", "1.0.0", deprecation_message)


✓ Official IP-Adapter loaded
processors: 32
<class 'diffusers.models.unets.unet_2d_condition.UNet2DConditionModel'>

Processors after IP-Adapter:
LoRAAttnProcessor2_0 : 16
LoRAIPAttnProcessor2_0 : 16

Processors with to_k_ip/to_v_ip: 16


### Dataset

In [6]:
# ============================================================
# Cell — Dataset + DataLoaders
# ============================================================

import os
import pickle
import pandas as pd
from torch.utils.data import DataLoader
from src.dataset import FIWTriadsDataset

EMBEDDINGS_PATH = f"{DRIVE}/data/embeddings/arcface_insightface.pkl"
FIW_ROOT        = f"{DRIVE}/data/raw/FIDs"
TRAIN_CSV       = f"{DRIVE}/data/splits/train_identity.csv"
VAL_CSV         = f"{DRIVE}/data/splits/val_identity.csv"
AGES_CSV        = f"{DRIVE}/data/ages/mivolo_ages.csv"
GENDER_CACHE    = f"{DRIVE}/data/gender_dict.pkl"
TEST_CSV = f"{DRIVE}/data/splits/test_identity.csv"

with open(EMBEDDINGS_PATH, "rb") as f:
    embeddings = pickle.load(f)

with open(GENDER_CACHE, "rb") as f:
    gender_dict = pickle.load(f)

ages_path = AGES_CSV if os.path.exists(AGES_CSV) else None

train_ds = FIWTriadsDataset(
    csv_path=TRAIN_CSV,
    embeddings=embeddings,
    fiw_root=FIW_ROOT,
    gender_dict=gender_dict,
    ages_csv_path=ages_path,
)

val_ds = FIWTriadsDataset(
    csv_path=VAL_CSV,
    embeddings=embeddings,
    fiw_root=FIW_ROOT,
    gender_dict=gender_dict,
    ages_csv_path=ages_path,
)

test_ds = FIWTriadsDataset(
    csv_path=TEST_CSV,
    embeddings=embeddings,
    fiw_root=FIW_ROOT,
    gender_dict=gender_dict,
    ages_csv_path=ages_path,
)

test_loader = DataLoader(
    test_ds,
    batch_size=1,
    shuffle=False,
    num_workers=0,
    pin_memory=False,
)

train_loader = DataLoader(
    train_ds,
    batch_size=1,
    shuffle=True,
    num_workers=0,
    pin_memory=False,
)

val_loader = DataLoader(
    val_ds,
    batch_size=1,
    shuffle=False,
    num_workers=0,
    pin_memory=False,
)

print(f"✓ Train: {len(train_ds):,} triads")
print(f"✓ Val:   {len(val_ds):,} triads")
print(f"✓ Test:  {len(test_ds):,} triads")

sample = test_ds[0]
print("Keys sample:", sample.keys())
print("img:", sample["img"].shape)
print("h_c:", sample["h_c"].shape)
print("age:", sample["age"])
print("gender:", sample["gender"])
print("key_p:", sample.get("key_p", "N/A"))
print("key_m:", sample.get("key_m", "N/A"))
print("key_c:", sample.get("key_c", "N/A"))

✓ Train: 1,563 triads
✓ Val:   196 triads
✓ Test:  208 triads
Keys sample: dict_keys(['h_p', 'h_m', 'h_c', 'img', 'age', 'gender', 'key_p', 'key_m', 'key_c'])
img: torch.Size([3, 512, 512])
h_c: torch.Size([512])
age: 43
gender: woman
key_p: F0003/MID2/P00019_face2.jpg
key_m: F0003/MID3/P00022_face1.jpg
key_c: F0003/MID1/P00019_face1.jpg


### Reload

In [ ]:
import os, sys, shutil, importlib

if os.path.exists("/content/src"):
    shutil.rmtree("/content/src")

shutil.copytree(f"{DRIVE}/src", "/content/src")

if "/content" not in sys.path:
    sys.path.insert(0, "/content")

for m in list(sys.modules.keys()):
    if m.startswith("src"):
        del sys.modules[m]

importlib.invalidate_caches()

from src.bap2 import BAP2
from src.att2 import attach_bap2

print("✓ BAP2 and att2 imported correctly")
print(BAP2)
print(attach_bap2)

### Attach

In [7]:
# ============================================================
# Re-attach BAP2 lineal + sanity check
# ============================================================


from src.bap2 import BAP2
from src.att2 import attach_bap2

print("✓ src recargado")
print("BAP2:", BAP2)
print("attach_bap2:", attach_bap2)

# Instalar BAP2 lineal
bap2_modules = attach_bap2(
    pipe.unet,
    num_tokens=4,
    scale=1.0,
    child_scale=1.0,
    parent_scale=0.1,
    gate_groups=16,
    gate_hidden=256,   # ignored by the linear BAP2 gate
    temperature=1.0,
    device=DEVICE,
    dtype=DTYPE,
    copy_ip_weights=True,
)

print("\n✓ attach BAP2 lineal terminado")
print("BAP2 modules:", len(bap2_modules))

# Contar processors
types = Counter(type(proc).__name__ for proc in pipe.unet.attn_processors.values())

print("\nProcessors actuales:")
for k, v in types.items():
    print(k, ":", v)

# Check dtype, parameters and alpha
gate_dtypes = Counter()
total_gate_params = 0
max_abs_w = 0.0
max_abs_b = 0.0

for name, proc in bap2_modules.items():
    gate_dtypes[str(proc.gate.weight.dtype)] += 1
    total_gate_params += sum(p.numel() for p in proc.gate.parameters())

    max_abs_w = max(
        max_abs_w,
        proc.gate.weight.detach().abs().max().float().item()
    )
    max_abs_b = max(
        max_abs_b,
        proc.gate.bias.detach().abs().max().float().item()
    )

print("\nDtypes gate.weight:")
for k, v in gate_dtypes.items():
    print(k, ":", v)

print("\nGate params totales:", total_gate_params)

print("\nGate initialization:")
print("max |W|:", max_abs_w)
print("max |b|:", max_abs_b)

# Dummy alpha of the first module
with torch.no_grad():
    first_name, first_proc = next(iter(bap2_modules.items()))

    in_features = first_proc.gate.in_features

    dummy = torch.randn(
        1,
        in_features,
        device=DEVICE,
        dtype=torch.float32,
    )

    alpha = torch.sigmoid(first_proc.gate(dummy))

print("\nDummy alpha, first module:")
print("module:", first_name)
print("gate input dim:", in_features)
print("alpha shape:", alpha.shape)
print("alpha mean:", alpha.mean().item())
print("alpha min :", alpha.min().item())
print("alpha max :", alpha.max().item())

print("\nVRAM usada:", torch.cuda.memory_allocated() / 1e9, "GB")

if (
    len(bap2_modules) == 16
    and str(first_proc.gate.weight.dtype) == "torch.float32"
    and max_abs_w == 0.0
    and max_abs_b == 0.0
):
    print("\n✓ Sanity check BAP2 lineal OK")
else:
    print("\n⚠ Check the linear BAP2 before continuing")

✓ src recargado
BAP2: <class 'src.bap2.BAP2'>
attach_bap2: <function attach_bap2 at 0x7c12310814e0>
✓ BAP2 instalado
  attn2 reemplazados:      16
  attn1/intactos:          16
  pesos copiados desde IP: 16/16
  pesos copiados desde BAP:0/16
  child_scale:             1.0
  parent_scale:            0.1
  gate_groups:             16
  gate_hidden:             256
  temperature:             1.0
  alpha inicial esperado:  0.5
  gate dtype esperado:     float32

✓ attach BAP2 lineal terminado
BAP2 modules: 16

Processors actuales:
LoRAAttnProcessor2_0 : 16
BAP2 : 16

Dtypes gate.weight:
torch.float32 : 16

Gate params totales: 792832

Gate initialization:
max |W|: 0.0
max |b|: 0.0

Dummy alpha, first module:
module: down_blocks_0_attentions_0_transformer_blocks_0_attn2
gate input dim: 2176
alpha shape: torch.Size([1, 16])
alpha mean: 0.5
alpha min : 0.5
alpha max : 0.5

VRAM usada: 2.50650624 GB

✓ Sanity check BAP2 lineal OK


### Load checkpoint

In [ ]:
# ============================================================
# Cell 7 — Load the latest BAP2 checkpoint + checks
# Requiere:
# bap2_modules, DEVICE, DRIVE
# ============================================================

import os, glob, torch
import pandas as pd

# ============================================================
# Use step600 DIRECTLY
# ============================================================

BAP2_CKPT_PATH = (
    f"{DRIVE}/results/child_anchor_bap_gate/"
    "bap2_linear_continue_step500_to_1000_lr1e-4/"
    "bap2_s600.pt"
)

print("Checkpoint seleccionado:")
print(BAP2_CKPT_PATH)
print("Existe:", os.path.exists(BAP2_CKPT_PATH))

assert os.path.exists(BAP2_CKPT_PATH), "BAP2 checkpoint does not exist."

# ------------------------------------------------------------
# 2. Load checkpoint
# ------------------------------------------------------------

ckpt = torch.load(BAP2_CKPT_PATH, map_location=DEVICE)

print("\nClaves checkpoint:", ckpt.keys())
print("Step checkpoint:", ckpt.get("step", "N/A"))

assert "bap2_gate_state" in ckpt, "The checkpoint has no bap2_gate_state."

gate_state = ckpt["bap2_gate_state"]

loaded = 0
missing_names = []

for name, proc in bap2_modules.items():
    if name in gate_state:
        proc.gate.load_state_dict(gate_state[name])
        proc.gate.to(device=DEVICE)
        proc.gate.float()
        loaded += 1
    else:
        missing_names.append(name)

print("\nGates loaded:", loaded, "/", len(bap2_modules))

if missing_names:
    print("Modules without state in checkpoint:")
    for n in missing_names:
        print(" -", n)

assert loaded == len(bap2_modules), "Not all BAP2 gates were loaded."

START_STEP = int(ckpt["step"])
TARGET_STEP = 1000

print("\nSTART_STEP:", START_STEP)
print("TARGET_STEP:", TARGET_STEP)

# ------------------------------------------------------------
# 3. Check parameters
# ------------------------------------------------------------

dtype_set = set()
nan_count = 0
max_abs = 0.0
param_std_vals = []

for name, proc in bap2_modules.items():
    dtype_set.add(str(proc.gate.weight.dtype))

    for p in proc.gate.parameters():
        nan_count += torch.isnan(p).sum().item()
        max_abs = max(max_abs, p.detach().abs().max().float().item())
        param_std_vals.append(p.detach().float().std().item())

print("\nParameter check:")
print("Dtypes gate:", dtype_set)
print("NaNs in gate:", nan_count)
print("max_abs gate params:", max_abs)
print("std promedio params:", sum(param_std_vals) / len(param_std_vals))

assert dtype_set == {"torch.float32"}, "The gate is not float32."
assert nan_count == 0, "There are NaNs in the gate parameters."
assert max_abs > 0.0, "The gate looks reset; the checkpoint did not load correctly."

# ------------------------------------------------------------
# 4. Verificar alpha dummy
# ------------------------------------------------------------

with torch.no_grad():
    first_name, first_proc = next(iter(bap2_modules.items()))

    dummy = torch.randn(
        1,
        first_proc.gate.in_features,
        device=DEVICE,
        dtype=torch.float32,
    )

    alpha = torch.sigmoid(first_proc.gate(dummy))

print("\nAlpha dummy checkpoint:")
print("module:", first_name)
print("alpha mean:", alpha.mean().item())
print("alpha min :", alpha.min().item())
print("alpha max :", alpha.max().item())
print("alpha std :", alpha.std().item())
print("alpha has NaN:", torch.isnan(alpha).any().item())

assert not torch.isnan(alpha).any().item(), "Alpha dummy tiene NaN."

# ------------------------------------------------------------
# 5. Load previous BAP2 λ=0.1 evaluation if it exists
# ------------------------------------------------------------

# ============================================================
# Use step600 DIRECTLY
# ============================================================

BAP2_CKPT_PATH = (
    f"{DRIVE}/results/child_anchor_bap_gate/"
    "bap2_linear_continue_step500_to_1000_lr1e-4/"
    "bap2_s600.pt"
)

# ------------------------------------------------------------
# 6. Verificar dataset
# ------------------------------------------------------------

required = ["train_ds", "val_ds"]
missing = [x for x in required if x not in globals()]
if missing:
    raise RuntimeError(f"Faltan datasets: {missing}")

sample = train_ds[0]

print("\nSample train_ds[0] keys:", sample.keys())
print("h_c:", sample["h_c"].shape)
print("h_p:", sample["h_p"].shape)
print("h_m:", sample["h_m"].shape)
print("img:", sample["img"].shape)
print("gender:", sample.get("gender", "N/A"))
print("age:", sample.get("age", "N/A"))

print("\n ALL OK: BAP2 checkpoint loaded and ready for training/evaluation.")

Checkpoint seleccionado:
/content/drive/MyDrive/Notebook4/results/child_anchor_bap_gate/bap2_linear_continue_step500_to_1000_lr1e-4/bap2_s600.pt
Existe: True

Claves checkpoint: dict_keys(['bap2_gate_state', 'step', 'config', 'logs'])
Step checkpoint: 600

Gates loaded: 16 / 16

START_STEP: 600
TARGET_STEP: 800

Parameter check:
Dtypes gate: {'torch.float32'}
NaNs in gate: 0
max_abs gate params: 0.00423230417072773
std promedio params: 0.0004358860115871721

Alpha dummy checkpoint:
module: down_blocks_0_attentions_0_transformer_blocks_0_attn2
alpha mean: 0.5001251101493835
alpha min : 0.49978432059288025
alpha max : 0.5015527606010437
alpha std : 0.0004014701407868415
alpha has NaN: False

Sample train_ds[0] keys: dict_keys(['h_p', 'h_m', 'h_c', 'img', 'age', 'gender', 'key_p', 'key_m', 'key_c'])
h_c: torch.Size([512])
h_p: torch.Size([512])
h_m: torch.Size([512])
img: torch.Size([3, 512, 512])
gender: man
age: 35

 ALL OK: BAP2 checkpoint loaded and ready for training/evaluation.


### Training

In [8]:
# ============================================================
# ENTRENAMIENTO FINAL BAP2 — step1000 → step4689
# 3 effective epochs over train_ds
#
# train_ds = 1563 triads
# batch_size = 1
# 1 epoch efectiva = 1563 steps
# 3 epochs efectivas = 4689 steps
#
# Trainable: ONLY the Heredity Gate
# Loss: L_diff
# ============================================================

import os
import torch
import torch.nn.functional as F
import pandas as pd
from tqdm.auto import tqdm
from datetime import datetime

# ------------------------------------------------------------
# Main configuration
# ------------------------------------------------------------

CKPT_START = f"{DRIVE}/results/ckpt/bap2_s1000.pt"

RUN_NAME = "bap2_4689"
OUT_DIR = f"{DRIVE}/results/ckpt/{RUN_NAME}"
os.makedirs(OUT_DIR, exist_ok=True)

LOGS_PATH = f"{OUT_DIR}/logs.csv"

START_STEP_EXPECTED = 1000
TARGET_STEP = 4689

SAVE_STEPS = {1563, 3126, 4689}
LOG_EVERY = 10

LR_BAP2 = 1e-4
WEIGHT_DECAY = 0.0
ADAM_EPS = 1e-6
MAX_GRAD_NORM = 1.0

PARENT_SCALE_TRAIN = 0.1

print("CKPT_START:", CKPT_START)
print("Existe:", os.path.exists(CKPT_START))
print("OUT_DIR:", OUT_DIR)
print("TARGET_STEP:", TARGET_STEP)
print("SAVE_STEPS:", SAVE_STEPS)

assert os.path.exists(CKPT_START), "The step1000 checkpoint does not exist."
assert "train_loader" in globals(), "Falta train_loader."
assert "bap2_modules" in globals(), "Falta bap2_modules."
assert len(bap2_modules) == 16, "bap2_modules must have 16 modules."

# ------------------------------------------------------------
# Load the step1000 checkpoint
# ------------------------------------------------------------

ckpt = torch.load(CKPT_START, map_location=DEVICE)

assert "bap2_gate_state" in ckpt, "Checkpoint without bap2_gate_state."
assert int(ckpt["step"]) == START_STEP_EXPECTED, f"Expected step1000, got {ckpt['step']}."

gate_state = ckpt["bap2_gate_state"]

loaded = 0
for name, proc in bap2_modules.items():
    assert name in gate_state, f"Missing gate_state for {name}"
    proc.gate.load_state_dict(gate_state[name])
    proc.gate.to(DEVICE)
    proc.gate.float()
    loaded += 1

print(f"✓ Cargado step{ckpt['step']} | gates {loaded}/{len(bap2_modules)}")

START_STEP = int(ckpt["step"])
CURRENT_STEP = START_STEP

# ------------------------------------------------------------
# Congelar todo excepto gate
# ------------------------------------------------------------

pipe.unet.train()
pipe.vae.eval()
pipe.text_encoder.eval()
ip_model.image_proj_model.eval()

for p in pipe.unet.parameters():
    p.requires_grad = False

for p in pipe.vae.parameters():
    p.requires_grad = False

for p in pipe.text_encoder.parameters():
    p.requires_grad = False

for p in ip_model.image_proj_model.parameters():
    p.requires_grad = False

bap2_gate_params = []

for name, proc in bap2_modules.items():
    proc.gate.float()
    for p in proc.gate.parameters():
        p.requires_grad = True
        bap2_gate_params.append(p)

print("Gate params entrenables:", sum(p.numel() for p in bap2_gate_params))
print("dtype gate example:", bap2_gate_params[0].dtype)

optimizer = torch.optim.AdamW(
    bap2_gate_params,
    lr=LR_BAP2,
    eps=ADAM_EPS,
    weight_decay=WEIGHT_DECAY,
)

# ------------------------------------------------------------
# Helpers
# ------------------------------------------------------------

def ensure_2d(x):
    if x.ndim == 1:
        x = x.unsqueeze(0)
    return x

def set_parent_scale_bap2(unet, value):
    for proc in unet.attn_processors.values():
        if type(proc).__name__ == "BAP2":
            proc.parent_scale = value

def collect_bap2_alpha_stats():
    vals = []

    for name, proc in bap2_modules.items():
        if getattr(proc, "last_alpha", None) is not None:
            vals.append(proc.last_alpha.detach().float().cpu())

    if len(vals) == 0:
        return {
            "alpha_mean": None,
            "alpha_min": None,
            "alpha_max": None,
            "alpha_std": None,
            "alpha_has_nan": None,
        }

    x = torch.cat(vals, dim=0)

    return {
        "alpha_mean": x.mean().item(),
        "alpha_min": x.min().item(),
        "alpha_max": x.max().item(),
        "alpha_std": x.std().item(),
        "alpha_has_nan": torch.isnan(x).any().item(),
    }

def bap2_param_stats():
    vals = []

    for name, proc in bap2_modules.items():
        for p in proc.gate.parameters():
            vals.append(p.detach().float().flatten().cpu())

    x = torch.cat(vals)

    return {
        "gate_param_mean": x.mean().item(),
        "gate_param_std": x.std().item(),
        "gate_param_max_abs": x.abs().max().item(),
        "gate_param_has_nan": torch.isnan(x).any().item(),
    }

def build_prompts(batch):
    ages = batch["age"]
    genders = batch["gender"]

    if torch.is_tensor(ages):
        ages = ages.detach().cpu().tolist()

    if isinstance(genders, str):
        genders = [genders]

    prompts = []

    for gender, age in zip(genders, ages):
        if gender not in ["man", "woman", "person"]:
            gender = "person"

        prompts.append(
            f"portrait photo of a {gender}, age {int(age)}, realistic, neutral expression"
        )

    return prompts

def train_step_bap2(batch):
    img = batch["img"].to(device=DEVICE, dtype=DTYPE)

    h_c = ensure_2d(batch["h_c"].to(device=DEVICE, dtype=DTYPE))
    h_p = ensure_2d(batch["h_p"].to(device=DEVICE, dtype=DTYPE))
    h_m = ensure_2d(batch["h_m"].to(device=DEVICE, dtype=DTYPE))

    prompts = build_prompts(batch)

    set_parent_scale_bap2(pipe.unet, PARENT_SCALE_TRAIN)

    with torch.no_grad():
        latents = pipe.vae.encode(img).latent_dist.sample()
        latents = latents * pipe.vae.config.scaling_factor

        noise = torch.randn_like(latents)

        timesteps = torch.randint(
            low=0,
            high=pipe.scheduler.config.num_train_timesteps,
            size=(latents.shape[0],),
            device=DEVICE,
        ).long()

        noisy_latents = pipe.scheduler.add_noise(latents, noise, timesteps)

        child_tokens = ip_model.image_proj_model(h_c)
        father_tokens = ip_model.image_proj_model(h_p)
        mother_tokens = ip_model.image_proj_model(h_m)

        prompt_embeds, _ = pipe.encode_prompt(
            prompt=prompts,
            device=DEVICE,
            num_images_per_prompt=1,
            do_classifier_free_guidance=False,
        )

        encoder_hidden_states = torch.cat(
            [prompt_embeds, child_tokens, father_tokens, mother_tokens],
            dim=1,
        )

    noise_pred = pipe.unet(
        noisy_latents,
        timesteps,
        encoder_hidden_states=encoder_hidden_states,
    ).sample

    loss = F.mse_loss(noise_pred.float(), noise.float())

    return loss

def save_bap2_checkpoint(global_step, logs):
    bap2_gate_state = {
        name: proc.gate.state_dict()
        for name, proc in bap2_modules.items()
    }

    ckpt_path = f"{OUT_DIR}/bap2_linear_step{global_step}.pt"

    torch.save(
        {
            "bap2_gate_state": bap2_gate_state,
            "step": int(global_step),
            "config": {
                "run_name": RUN_NAME,
                "loss": "L_diff",
                "start_step": START_STEP,
                "target_step": TARGET_STEP,
                "effective_epochs_target": TARGET_STEP / len(train_ds),
                "train_size": len(train_ds),
                "batch_size": 1,
                "lr_bap2": LR_BAP2,
                "adam_eps": ADAM_EPS,
                "weight_decay": WEIGHT_DECAY,
                "max_grad_norm": MAX_GRAD_NORM,
                "parent_scale": PARENT_SCALE_TRAIN,
                "child_scale": 1.0,
                "gate": "contextual_linear",
                "gate_dtype": "float32",
                "trainable": "gate_only",
                "timestamp": datetime.now().isoformat(),
            },
            "logs": logs,
        },
        ckpt_path,
    )

    return ckpt_path

# ------------------------------------------------------------
# Resume logs if they exist
# Note: this does NOT resume the optimizer, it only avoids losing logs.
# To resume from another checkpoint, load it manually.
# ------------------------------------------------------------

logs_continue = []
global_step = START_STEP

print("\nEntrenamiento:")
print("START_STEP:", START_STEP)
print("TARGET_STEP:", TARGET_STEP)
print("epochs efectivas objetivo:", TARGET_STEP / len(train_ds))

loader_iter = iter(train_loader)

pbar = tqdm(total=TARGET_STEP - START_STEP)

# ------------------------------------------------------------
# Loop principal
# ------------------------------------------------------------

while global_step < TARGET_STEP:

    try:
        batch = next(loader_iter)
    except StopIteration:
        loader_iter = iter(train_loader)
        batch = next(loader_iter)

    batch = {
        k: v.to(DEVICE) if isinstance(v, torch.Tensor) else v
        for k, v in batch.items()
    }

    optimizer.zero_grad(set_to_none=True)

    loss = train_step_bap2(batch)

    if not torch.isfinite(loss):
        print("⚠ Non-finite loss at step", global_step + 1, "loss:", loss.item())
        break

    loss.backward()

    grad_norm = torch.nn.utils.clip_grad_norm_(
        bap2_gate_params,
        max_norm=MAX_GRAD_NORM,
    )

    optimizer.step()

    global_step += 1

    alpha_stats = collect_bap2_alpha_stats()
    param_stats = bap2_param_stats()

    row = {
        "global_step": global_step,
        "effective_epoch": global_step / len(train_ds),
        "loss": loss.item(),
        "grad_norm": float(grad_norm),
        **alpha_stats,
        **param_stats,
    }

    logs_continue.append(row)

    if global_step % LOG_EVERY == 0 or global_step in SAVE_STEPS or global_step == START_STEP + 1:
        print(
            f"step {global_step:04d} | "
            f"epoch={global_step / len(train_ds):.3f} | "
            f"loss={loss.item():.6f} | "
            f"grad={float(grad_norm):.6e} | "
            f"alpha_mean={alpha_stats['alpha_mean']:.6f} | "
            f"alpha_std={alpha_stats['alpha_std']:.6f} | "
            f"alpha_min={alpha_stats['alpha_min']:.6f} | "
            f"alpha_max={alpha_stats['alpha_max']:.6f} | "
            f"param_max_abs={param_stats['gate_param_max_abs']:.6e}"
        )

    # ----------------------------
    # Seguridad
    # ----------------------------
    if alpha_stats["alpha_has_nan"] or param_stats["gate_param_has_nan"]:
        print("⚠ NaN detected in alpha or parameters. Stopping.")
        break

    # Allow the gate to open while avoiding extreme collapse.
    if alpha_stats["alpha_min"] < 0.20 or alpha_stats["alpha_max"] > 0.80:
        print("⚠ Gate demasiado extremo. Deteniendo.")
        print(alpha_stats)
        break

    # Evitar sesgo global fuerte hacia un progenitor.
    if alpha_stats["alpha_mean"] < 0.42 or alpha_stats["alpha_mean"] > 0.58:
        print("⚠ Global father/mother bias detected. Stopping.")
        print(alpha_stats)
        break

    # ----------------------------
    # Save at key checkpoints
    # ----------------------------
    if global_step in SAVE_STEPS:
        ckpt_path = save_bap2_checkpoint(global_step, logs_continue)
        print(f"✓ Checkpoint saved: {ckpt_path}")

    # Save logs incrementally every 50 steps
    if global_step % 50 == 0:
        pd.DataFrame(logs_continue).to_csv(LOGS_PATH, index=False)

    pbar.update(1)

pbar.close()

# ------------------------------------------------------------
# Final save
# ------------------------------------------------------------

df_logs_final_train = pd.DataFrame(logs_continue)
df_logs_final_train.to_csv(LOGS_PATH, index=False)

final_ckpt_path = save_bap2_checkpoint(global_step, logs_continue)

CURRENT_STEP = global_step
START_STEP = global_step

print("\n✓ Entrenamiento final terminado")
print("Last global_step:", global_step)
print("Epoch efectiva:", global_step / len(train_ds))
print("Logs:", LOGS_PATH)
print("Checkpoint final:", final_ckpt_path)
print("CURRENT_STEP:", CURRENT_STEP)

if len(df_logs_final_train) > 0:
    print("\nAlpha summary:")
    print("alpha_mean inicial:", df_logs_final_train.iloc[0]["alpha_mean"])
    print("alpha_mean final:  ", df_logs_final_train.iloc[-1]["alpha_mean"])
    print("alpha_std inicial: ", df_logs_final_train.iloc[0]["alpha_std"])
    print("alpha_std final:   ", df_logs_final_train.iloc[-1]["alpha_std"])
    print("alpha_min final:   ", df_logs_final_train.iloc[-1]["alpha_min"])
    print("alpha_max final:   ", df_logs_final_train.iloc[-1]["alpha_max"])

    print("\nLoss summary:")
    print("loss inicial:", df_logs_final_train.iloc[0]["loss"])
    print("loss final:  ", df_logs_final_train.iloc[-1]["loss"])

display(df_logs_final_train.tail())

CKPT_START: /content/drive/MyDrive/Notebook4/results/ckpt/bap2_s1000.pt
Existe: True
OUT_DIR: /content/drive/MyDrive/Notebook4/results/ckpt/bap2_4689
TARGET_STEP: 4689
SAVE_STEPS: {4689, 1563, 3126}
✓ Cargado step1000 | gates 16/16
Gate params entrenables: 792832
dtype gate example: torch.float32

Entrenamiento:
START_STEP: 1000
TARGET_STEP: 4689
epochs efectivas objetivo: 3.0


  0%|          | 0/3689 [00:00<?, ?it/s]

step 1001 | epoch=0.640 | loss=0.074456 | grad=1.549425e-04 | alpha_mean=0.499737 | alpha_std=0.009128 | alpha_min=0.453669 | alpha_max=0.533014 | param_max_abs=7.395222e-03
step 1010 | epoch=0.646 | loss=0.660909 | grad=9.415825e-05 | alpha_mean=0.499533 | alpha_std=0.007303 | alpha_min=0.458652 | alpha_max=0.526375 | param_max_abs=7.202412e-03
step 1020 | epoch=0.653 | loss=0.483562 | grad=1.072951e-04 | alpha_mean=0.499483 | alpha_std=0.007600 | alpha_min=0.462143 | alpha_max=0.530738 | param_max_abs=7.179497e-03
step 1030 | epoch=0.659 | loss=0.047552 | grad=3.902100e-05 | alpha_mean=0.499436 | alpha_std=0.007725 | alpha_min=0.453226 | alpha_max=0.524300 | param_max_abs=7.362105e-03
step 1040 | epoch=0.665 | loss=0.005667 | grad=5.857036e-07 | alpha_mean=0.500407 | alpha_std=0.012379 | alpha_min=0.427096 | alpha_max=0.562601 | param_max_abs=7.398665e-03
step 1050 | epoch=0.672 | loss=0.083753 | grad=1.332768e-04 | alpha_mean=0.499614 | alpha_std=0.008214 | alpha_min=0.457727 | alph

,global_step,effective_epoch,loss,grad_norm,alpha_mean,alpha_min,alpha_max,alpha_std,alpha_has_nan,gate_param_mean,gate_param_std,gate_param_max_abs,gate_param_has_nan
3684,4685,2.997441,0.018683,2.121083e-05,0.497536,0.368891,0.650033,0.031738,False,0.000004,0.001250,0.022381,False
3685,4686,2.998081,0.274145,2.532855e-04,0.498717,0.359291,0.731647,0.033033,False,0.000004,0.001251,0.022384,False
3686,4687,2.998720,0.004652,1.739100e-07,0.498514,0.364390,0.626687,0.030655,False,0.000004,0.001251,0.022388,False
3687,4688,2.999360,0.338190,1.961141e-04,0.497441,0.368342,0.605738,0.027600,False,0.000004,0.001251,0.022425,False
3688,4689,3.000000,0.013843,4.963456e-05,0.497532,0.343440,0.627004,0.030428,False,0.000004,0.001252,0.022458,False


### Evaluation

In [9]:
# ============================================================
# FINAL CHECKPOINT SELECTION
# Evaluates step1000, step1563, step3126, step4689
# 10 VAL triads × age 25 × λ=0.1
# ============================================================

import os
import torch
import torch.nn.functional as F
import torchvision.transforms as T
import pandas as pd
from torch.utils.data._utils.collate import default_collate
from IPython.display import display

# ------------------------------------------------------------
# Checkpoints candidatos
# ------------------------------------------------------------

CKPT_CANDIDATES = {
    "step1000": f"{DRIVE}/results/ckpt/bap2_s1000.pt",

    "step1563": (
        f"{DRIVE}/results/ckpt/"
        "bap2_4689/"
        "bap2_s1563.pt"
    ),

    "step3126": (
        f"{DRIVE}/results/ckpt/"
        "bap2_4689/"
        "bap2_s3126.pt"
    ),

    "step4689": (
        f"{DRIVE}/results/ckpt/"
        "bap2_4689/"
        "bap2_s4689.pt"
    ),
}

available_ckpts = {}

for name, path in CKPT_CANDIDATES.items():
    exists = os.path.exists(path)
    print(f"{name}: {exists} | {path}")
    if exists:
        available_ckpts[name] = path

assert len(available_ckpts) >= 2, "At least 2 checkpoints are needed for comparison."

# ------------------------------------------------------------
# Configuration
# ------------------------------------------------------------

N_TRIADS = 10
EVAL_IDXS = [179, 12, 35, 48, 61, 74, 89, 103, 128, 150]
EVAL_IDXS = EVAL_IDXS[:N_TRIADS]

AGE = 25
PARENT_SCALE = 0.1

GUIDANCE = 4.0
STEPS = 30
BASE_SEED = 42

NEG_PROMPT = "monochrome, lowres, bad anatomy, worst quality, low quality, blurry"

OUT_DIR = f"{DRIVE}/results/checkpoint_selection_final"
os.makedirs(OUT_DIR, exist_ok=True)

print("\nEVAL_IDXS:", EVAL_IDXS)
print("AGE:", AGE)
print("PARENT_SCALE:", PARENT_SCALE)
print("OUT_DIR:", OUT_DIR)

# ------------------------------------------------------------
# Helpers
# ------------------------------------------------------------

preprocess_arcface = T.Compose([
    T.Resize((112, 112)),
    T.ToTensor(),
    T.Normalize(mean=[0.5, 0.5, 0.5], std=[0.5, 0.5, 0.5]),
])

@torch.inference_mode()
def arcface_embed_from_pil(img):
    x = preprocess_arcface(img.convert("RGB")).unsqueeze(0).to(DEVICE).float()
    emb = arcface_torch(x)
    emb = F.normalize(emb, p=2, dim=1)
    return emb

def ensure_2d(x):
    if x.ndim == 1:
        x = x.unsqueeze(0)
    return x

def set_parent_scale_bap2(value):
    for proc in pipe.unet.attn_processors.values():
        if type(proc).__name__ == "BAP2":
            proc.parent_scale = value

def collect_bap2_alpha_stats():
    vals = []

    for proc in pipe.unet.attn_processors.values():
        if type(proc).__name__ == "BAP2":
            if getattr(proc, "last_alpha", None) is not None:
                vals.append(proc.last_alpha.detach().float().cpu())

    if len(vals) == 0:
        return {
            "alpha_mean": None,
            "alpha_std": None,
            "alpha_min": None,
            "alpha_max": None,
        }

    x = torch.cat(vals, dim=0)

    return {
        "alpha_mean": x.mean().item(),
        "alpha_std": x.std().item(),
        "alpha_min": x.min().item(),
        "alpha_max": x.max().item(),
    }

def load_bap2_checkpoint(ckpt_path):
    ckpt = torch.load(ckpt_path, map_location=DEVICE)

    assert "bap2_gate_state" in ckpt, "Checkpoint without bap2_gate_state."

    gate_state = ckpt["bap2_gate_state"]

    loaded = 0

    for name, proc in bap2_modules.items():
        assert name in gate_state, f"Missing gate for {name}"
        proc.gate.load_state_dict(gate_state[name])
        proc.gate.to(DEVICE)
        proc.gate.float()
        loaded += 1

    print(f"✓ Cargado checkpoint step={ckpt.get('step')} | gates={loaded}/{len(bap2_modules)}")
    return int(ckpt.get("step", -1))

@torch.inference_mode()
def generate_bap2_eval(h_c, h_p, h_m, gender, age, parent_scale, seed):
    pipe.unet.eval()
    pipe.vae.eval()
    pipe.text_encoder.eval()
    ip_model.image_proj_model.eval()

    set_parent_scale_bap2(parent_scale)

    h_c = ensure_2d(h_c).to(DEVICE, dtype=DTYPE)
    h_p = ensure_2d(h_p).to(DEVICE, dtype=DTYPE)
    h_m = ensure_2d(h_m).to(DEVICE, dtype=DTYPE)

    child_tokens  = ip_model.image_proj_model(h_c)
    father_tokens = ip_model.image_proj_model(h_p)
    mother_tokens = ip_model.image_proj_model(h_m)

    zero_child_tokens  = ip_model.image_proj_model(torch.zeros_like(h_c))
    zero_father_tokens = ip_model.image_proj_model(torch.zeros_like(h_p))
    zero_mother_tokens = ip_model.image_proj_model(torch.zeros_like(h_m))

    prompt = f"portrait photo of a {gender}, age {age}, realistic, neutral expression"

    prompt_embeds, negative_prompt_embeds = pipe.encode_prompt(
        prompt=[prompt],
        device=DEVICE,
        num_images_per_prompt=1,
        do_classifier_free_guidance=True,
        negative_prompt=[NEG_PROMPT],
    )

    prompt_embeds = torch.cat(
        [prompt_embeds, child_tokens, father_tokens, mother_tokens],
        dim=1,
    )

    negative_prompt_embeds = torch.cat(
        [negative_prompt_embeds, zero_child_tokens, zero_father_tokens, zero_mother_tokens],
        dim=1,
    )

    generator = torch.Generator(device=DEVICE).manual_seed(seed)

    img = pipe(
        prompt_embeds=prompt_embeds,
        negative_prompt_embeds=negative_prompt_embeds,
        guidance_scale=GUIDANCE,
        num_inference_steps=STEPS,
        generator=generator,
    ).images[0]

    alpha_stats = collect_bap2_alpha_stats()

    return img, alpha_stats

# ============================================================
# Evaluation loop
# ============================================================

rows = []

for ckpt_name, ckpt_path in available_ckpts.items():

    print("\n" + "=" * 80)
    print("CHECKPOINT:", ckpt_name)
    print("=" * 80)

    loaded_step = load_bap2_checkpoint(ckpt_path)

    for triad_num, idx in enumerate(EVAL_IDXS, start=1):

        sample = val_ds[idx]

        batch = default_collate([sample])
        batch = {
            k: v.to(DEVICE) if isinstance(v, torch.Tensor) else v
            for k, v in batch.items()
        }

        h_c = ensure_2d(batch["h_c"])
        h_p = ensure_2d(batch["h_p"])
        h_m = ensure_2d(batch["h_m"])

        h_c_eval = F.normalize(h_c.float(), p=2, dim=1)
        h_p_eval = F.normalize(h_p.float(), p=2, dim=1)
        h_m_eval = F.normalize(h_m.float(), p=2, dim=1)

        gender = sample.get("gender", "person")
        if gender not in ["man", "woman", "person"]:
            gender = "person"

        seed = BASE_SEED + idx * 1000 + AGE

        img, alpha_stats = generate_bap2_eval(
            h_c=h_c,
            h_p=h_p,
            h_m=h_m,
            gender=gender,
            age=AGE,
            parent_scale=PARENT_SCALE,
            seed=seed,
        )

        h_gen = arcface_embed_from_pil(img)

        S_id = F.cosine_similarity(h_gen, h_c_eval).item()
        S_p  = F.cosine_similarity(h_gen, h_p_eval).item()
        S_m  = F.cosine_similarity(h_gen, h_m_eval).item()

        rows.append({
            "checkpoint": ckpt_name,
            "step": loaded_step,
            "triad_idx": idx,
            "triad_num": triad_num,
            "age": AGE,
            "parent_scale": PARENT_SCALE,
            "gender": gender,
            "S_id_child": S_id,
            "S_p_father": S_p,
            "S_m_mother": S_m,
            "S_family": 0.5 * (S_p + S_m),
            "balance_abs_gap": abs(S_p - S_m),
            "alpha_mean": alpha_stats["alpha_mean"],
            "alpha_std": alpha_stats["alpha_std"],
            "alpha_min": alpha_stats["alpha_min"],
            "alpha_max": alpha_stats["alpha_max"],
            "key_p": sample.get("key_p", ""),
            "key_m": sample.get("key_m", ""),
            "key_c": sample.get("key_c", ""),
            "seed": seed,
        })

        print(
            f"{ckpt_name} | idx={idx} | "
            f"S_id={S_id:.4f} | "
            f"S_family={0.5 * (S_p + S_m):.4f} | "
            f"gap={abs(S_p - S_m):.4f} | "
            f"alpha_std={alpha_stats['alpha_std']:.5f}"
        )

df_ckpt_selection = pd.DataFrame(rows)

print("\n==============================")
print("Full results")
print("==============================")
display(df_ckpt_selection)

# ============================================================
# Summary by checkpoint
# ============================================================

summary_ckpt = (
    df_ckpt_selection
    .groupby("checkpoint")
    [[
        "S_id_child",
        "S_p_father",
        "S_m_mother",
        "S_family",
        "balance_abs_gap",
        "alpha_mean",
        "alpha_std",
        "alpha_min",
        "alpha_max",
    ]]
    .mean()
    .reset_index()
)

print("\n==============================")
print("Summary by checkpoint")
print("==============================")
display(summary_ckpt)

# ============================================================
# Delta vs step1000 if it exists
# ============================================================

metrics = ["S_id_child", "S_p_father", "S_m_mother", "S_family", "balance_abs_gap"]

delta_rows = []

if "step1000" in df_ckpt_selection["checkpoint"].unique():

    base = df_ckpt_selection[df_ckpt_selection["checkpoint"] == "step1000"]

    for ckpt_name in sorted(df_ckpt_selection["checkpoint"].unique()):
        if ckpt_name == "step1000":
            continue

        comp = df_ckpt_selection[df_ckpt_selection["checkpoint"] == ckpt_name]

        merged = base[["triad_idx", "age"] + metrics].merge(
            comp[["triad_idx", "age"] + metrics],
            on=["triad_idx", "age"],
            suffixes=("_1000", f"_{ckpt_name}")
        )

        for m in metrics:
            d = merged[f"{m}_{ckpt_name}"] - merged[f"{m}_1000"]

            delta_rows.append({
                "checkpoint_vs_step1000": ckpt_name,
                "metric": m,
                "mean_delta": d.mean(),
                "median_delta": d.median(),
                "win_rate_gt_step1000": (d > 0).mean(),
                "n": len(d),
            })

df_delta_vs_1000 = pd.DataFrame(delta_rows)

print("\n==============================")
print("Delta contra step1000")
print("==============================")
display(df_delta_vs_1000)

# ============================================================
# Save
# ============================================================

df_ckpt_selection.to_csv(f"{OUT_DIR}/df_ckpt_selection_val10.csv", index=False)
summary_ckpt.to_csv(f"{OUT_DIR}/summary_ckpt_selection_val10.csv", index=False)
df_delta_vs_1000.to_csv(f"{OUT_DIR}/delta_vs_step1000_val10.csv", index=False)

print("\n✓ Saved to:")
print(OUT_DIR)

step1000: True | /content/drive/MyDrive/Notebook4/results/ckpt/bap2_s1000.pt
step1563: True | /content/drive/MyDrive/Notebook4/results/ckpt/bap2_4689/bap2_s1563.pt
step3126: True | /content/drive/MyDrive/Notebook4/results/ckpt/bap2_4689/bap2_s3126.pt
step4689: True | /content/drive/MyDrive/Notebook4/results/ckpt/bap2_4689/bap2_s4689.pt

EVAL_IDXS: [179, 12, 35, 48, 61, 74, 89, 103, 128, 150]
AGE: 25
PARENT_SCALE: 0.1
OUT_DIR: /content/drive/MyDrive/Notebook4/results/checkpoint_selection_final

CHECKPOINT: step1000
✓ Cargado checkpoint step=1000 | gates=16/16


  0%|          | 0/30 [00:00<?, ?it/s]

step1000 | idx=179 | S_id=-0.0285 | S_family=0.0037 | gap=0.0002 | alpha_std=0.00761


  0%|          | 0/30 [00:00<?, ?it/s]

step1000 | idx=12 | S_id=0.0737 | S_family=-0.0374 | gap=0.0135 | alpha_std=0.00773


...


Summary by checkpoint


,checkpoint,S_id_child,S_p_father,S_m_mother,S_family,balance_abs_gap,alpha_mean,alpha_std,alpha_min,alpha_max
0,step1000,0.115088,0.016306,-0.017194,-0.000444,0.051437,0.499733,0.007840,0.457910,0.530406
1,step1563,0.115169,0.015952,-0.017566,-0.000807,0.051655,0.499667,0.011526,0.432997,0.545513
2,step3126,0.115548,0.015895,-0.017455,-0.000780,0.051488,0.499636,0.018675,0.397413,0.592532
3,step4689,0.116182,0.015807,-0.017295,-0.000744,0.051078,0.498207,0.026405,0.374008,0.634236



Delta contra step1000


,checkpoint_vs_step1000,metric,mean_delta,median_delta,win_rate_gt_step1000,n
0,step1563,S_id_child,0.000081,-0.000221,0.4,10
1,step1563,S_p_father,-0.000354,-0.000141,0.3,10
2,step1563,S_m_mother,-0.000373,-0.000218,0.2,10
3,step1563,S_family,-0.000363,-0.000159,0.1,10
4,step1563,balance_abs_gap,0.000218,0.000337,0.6,10
5,step3126,S_id_child,0.000460,-0.000089,0.5,10
6,step3126,S_p_father,-0.000411,-0.000338,0.3,10
7,step3126,S_m_mother,-0.000262,-0.000258,0.3,10
8,step3126,S_family,-0.000336,-0.000110,0.3,10
9,step3126,balance_abs_gap,0.000051,-0.000328,0.3,10



✓ Saved to:
/content/drive/MyDrive/Notebook4/results/checkpoint_selection_final


In [10]:
# ============================================================
# TEST FINAL — BAP2 step4689
# 208 triads × 5 ages × 2 lambdas = 2080 images
#
# Methods:
#   BAP2 λ=0.0
#   BAP2 λ=0.1
#
# Guarda:
#   - CSV incremental
#   - generated images
#   - summary final
# Automatic resume if interrupted
# ============================================================

import os
import torch
import torch.nn.functional as F
import torchvision.transforms as T
import pandas as pd
from torch.utils.data._utils.collate import default_collate
from IPython.display import display

# ------------------------------------------------------------
# Configuration
# ------------------------------------------------------------

CKPT_FINAL = (
    f"{DRIVE}/results/ckpt/"
    "bap2_4689/"
    "bap2_s4689.pt"
)

assert os.path.exists(CKPT_FINAL), f"Final checkpoint does not exist: {CKPT_FINAL}"

AGES = [5, 15, 25, 40, 60]
PARENT_SCALES = [0.0, 0.1]

GUIDANCE = 4.0
STEPS = 30
BASE_SEED = 42

NEG_PROMPT = "monochrome, lowres, bad anatomy, worst quality, low quality, blurry"

OUT_DIR = f"{DRIVE}/results/eval/bap2"
IMG_DIR = f"{OUT_DIR}/images"
os.makedirs(OUT_DIR, exist_ok=True)
os.makedirs(IMG_DIR, exist_ok=True)

OUT_CSV = f"{OUT_DIR}/bap2.csv"

print("Checkpoint:", CKPT_FINAL)
print("Test triads:", len(test_ds))
print("Ages:", AGES)
print("Parent scales:", PARENT_SCALES)
print("Total images:", len(test_ds) * len(AGES) * len(PARENT_SCALES))
print("OUT_DIR:", OUT_DIR)

# ------------------------------------------------------------
# Load BAP2 checkpoint step4689
# ------------------------------------------------------------

ckpt = torch.load(CKPT_FINAL, map_location=DEVICE)

assert "bap2_gate_state" in ckpt, "Checkpoint without bap2_gate_state."
assert int(ckpt["step"]) == 4689, f"Expected step4689, got {ckpt['step']}."
assert "bap2_modules" in globals(), "Falta bap2_modules."
assert len(bap2_modules) == 16, "bap2_modules must have 16 modules."

gate_state = ckpt["bap2_gate_state"]

loaded = 0
for name, proc in bap2_modules.items():
    assert name in gate_state, f"Missing gate for {name}"
    proc.gate.load_state_dict(gate_state[name])
    proc.gate.to(DEVICE)
    proc.gate.float()
    loaded += 1

CURRENT_STEP = int(ckpt["step"])

print(f"✓ Checkpoint loaded: step={CURRENT_STEP}, gates={loaded}/{len(bap2_modules)}")

# ------------------------------------------------------------
# Helpers
# ------------------------------------------------------------

preprocess_arcface = T.Compose([
    T.Resize((112, 112)),
    T.ToTensor(),
    T.Normalize(mean=[0.5, 0.5, 0.5], std=[0.5, 0.5, 0.5]),
])

@torch.inference_mode()
def arcface_embed_from_pil(img):
    x = preprocess_arcface(img.convert("RGB")).unsqueeze(0).to(DEVICE).float()
    emb = arcface_torch(x)
    emb = F.normalize(emb, p=2, dim=1)
    return emb

def ensure_2d(x):
    if x.ndim == 1:
        x = x.unsqueeze(0)
    return x

def set_parent_scale_bap2(value):
    for proc in pipe.unet.attn_processors.values():
        if type(proc).__name__ == "BAP2":
            proc.parent_scale = value

def collect_bap2_alpha_stats():
    vals = []

    for proc in pipe.unet.attn_processors.values():
        if type(proc).__name__ == "BAP2":
            if getattr(proc, "last_alpha", None) is not None:
                vals.append(proc.last_alpha.detach().float().cpu())

    if len(vals) == 0:
        return {
            "alpha_mean": None,
            "alpha_std": None,
            "alpha_min": None,
            "alpha_max": None,
        }

    x = torch.cat(vals, dim=0)

    return {
        "alpha_mean": x.mean().item(),
        "alpha_std": x.std().item(),
        "alpha_min": x.min().item(),
        "alpha_max": x.max().item(),
    }

@torch.inference_mode()
def generate_bap2_final(h_c, h_p, h_m, gender, age, parent_scale, seed):
    pipe.unet.eval()
    pipe.vae.eval()
    pipe.text_encoder.eval()
    ip_model.image_proj_model.eval()

    set_parent_scale_bap2(parent_scale)

    h_c = ensure_2d(h_c).to(DEVICE, dtype=DTYPE)
    h_p = ensure_2d(h_p).to(DEVICE, dtype=DTYPE)
    h_m = ensure_2d(h_m).to(DEVICE, dtype=DTYPE)

    child_tokens  = ip_model.image_proj_model(h_c)
    father_tokens = ip_model.image_proj_model(h_p)
    mother_tokens = ip_model.image_proj_model(h_m)

    zero_child_tokens  = ip_model.image_proj_model(torch.zeros_like(h_c))
    zero_father_tokens = ip_model.image_proj_model(torch.zeros_like(h_p))
    zero_mother_tokens = ip_model.image_proj_model(torch.zeros_like(h_m))

    prompt = f"portrait photo of a {gender}, age {age}, realistic, neutral expression"

    prompt_embeds, negative_prompt_embeds = pipe.encode_prompt(
        prompt=[prompt],
        device=DEVICE,
        num_images_per_prompt=1,
        do_classifier_free_guidance=True,
        negative_prompt=[NEG_PROMPT],
    )

    prompt_embeds = torch.cat(
        [prompt_embeds, child_tokens, father_tokens, mother_tokens],
        dim=1,
    )

    negative_prompt_embeds = torch.cat(
        [negative_prompt_embeds, zero_child_tokens, zero_father_tokens, zero_mother_tokens],
        dim=1,
    )

    generator = torch.Generator(device=DEVICE).manual_seed(seed)

    img = pipe(
        prompt_embeds=prompt_embeds,
        negative_prompt_embeds=negative_prompt_embeds,
        guidance_scale=GUIDANCE,
        num_inference_steps=STEPS,
        generator=generator,
    ).images[0]

    alpha_stats = collect_bap2_alpha_stats()

    return img, alpha_stats, prompt

def save_generated_image(img, triad_idx, age, parent_scale):
    scale_tag = str(parent_scale).replace(".", "p")
    subdir = f"{IMG_DIR}/lambda_{scale_tag}/age_{age}"
    os.makedirs(subdir, exist_ok=True)

    path = f"{subdir}/triad_{triad_idx:04d}_age_{age}_lambda_{scale_tag}.jpg"
    img.save(path, quality=95)
    return path

# ------------------------------------------------------------
# Automatic resume
# ------------------------------------------------------------

rows = []
done = set()

if os.path.exists(OUT_CSV):
    old = pd.read_csv(OUT_CSV)
    rows = old.to_dict("records")
    done = set(zip(old["triad_idx"], old["age"], old["parent_scale"]))
    print(f"✓ Resume detected: {len(old)} existing rows")

# ------------------------------------------------------------
# Loop principal TEST
# ------------------------------------------------------------

for triad_idx in range(len(test_ds)):

    sample = test_ds[triad_idx]

    batch = default_collate([sample])
    batch = {
        k: v.to(DEVICE) if isinstance(v, torch.Tensor) else v
        for k, v in batch.items()
    }

    h_c = ensure_2d(batch["h_c"])
    h_p = ensure_2d(batch["h_p"])
    h_m = ensure_2d(batch["h_m"])

    h_c_eval = F.normalize(h_c.float(), p=2, dim=1)
    h_p_eval = F.normalize(h_p.float(), p=2, dim=1)
    h_m_eval = F.normalize(h_m.float(), p=2, dim=1)

    gender = sample.get("gender", "person")
    if gender not in ["man", "woman", "person"]:
        gender = "person"

    print(f"\nTriada {triad_idx + 1}/{len(test_ds)} | gender={gender}")

    for age in AGES:

        # Same seed for λ=0.0 and λ=0.1 on the same triad/age.
        paired_seed = BASE_SEED + triad_idx * 1000 + age

        for parent_scale in PARENT_SCALES:

            if (triad_idx, age, parent_scale) in done:
                print(f"skip triad={triad_idx} age={age} λ={parent_scale}")
                continue

            img, alpha_stats, prompt = generate_bap2_final(
                h_c=h_c,
                h_p=h_p,
                h_m=h_m,
                gender=gender,
                age=age,
                parent_scale=parent_scale,
                seed=paired_seed,
            )

            img_path = save_generated_image(
                img=img,
                triad_idx=triad_idx,
                age=age,
                parent_scale=parent_scale,
            )

            h_gen = arcface_embed_from_pil(img)

            S_id = F.cosine_similarity(h_gen, h_c_eval).item()
            S_p  = F.cosine_similarity(h_gen, h_p_eval).item()
            S_m  = F.cosine_similarity(h_gen, h_m_eval).item()

            row = {
                "variant": "BAP2_step4689",
                "checkpoint_step": CURRENT_STEP,
                "triad_idx": triad_idx,
                "age": age,
                "parent_scale": parent_scale,
                "gender": gender,
                "S_id_child": S_id,
                "S_p_father": S_p,
                "S_m_mother": S_m,
                "S_family": 0.5 * (S_p + S_m),
                "balance_abs_gap": abs(S_p - S_m),
                "alpha_mean": alpha_stats["alpha_mean"],
                "alpha_std": alpha_stats["alpha_std"],
                "alpha_min": alpha_stats["alpha_min"],
                "alpha_max": alpha_stats["alpha_max"],
                "key_p": sample.get("key_p", ""),
                "key_m": sample.get("key_m", ""),
                "key_c": sample.get("key_c", ""),
                "seed": paired_seed,
                "prompt": prompt,
                "image_path": img_path,
            }

            rows.append(row)

            # Incremental saving so progress is not lost
            pd.DataFrame(rows).to_csv(OUT_CSV, index=False)

            print(
                f"age={age} | λ={parent_scale} | "
                f"S_id={S_id:.4f} | "
                f"S_p={S_p:.4f} | "
                f"S_m={S_m:.4f} | "
                f"S_family={row['S_family']:.4f} | "
                f"gap={row['balance_abs_gap']:.4f} | "
                f"alpha_std={alpha_stats['alpha_std']:.5f}"
            )

# ------------------------------------------------------------
# Final summary BAP2
# ------------------------------------------------------------

df_final_bap2 = pd.DataFrame(rows)

print("\n==============================")
print("Filas finales BAP2")
print("==============================")
print(len(df_final_bap2))

expected_rows = len(test_ds) * len(AGES) * len(PARENT_SCALES)
print("Esperadas:", expected_rows)

if len(df_final_bap2) != expected_rows:
    print("⚠ Warning: unexpected number of rows. Check whether there was an interruption.")
else:
    print("✓ Row count is correct.")

summary_bap2_global = (
    df_final_bap2
    .groupby("parent_scale")
    [[
        "S_id_child",
        "S_p_father",
        "S_m_mother",
        "S_family",
        "balance_abs_gap",
        "alpha_mean",
        "alpha_std",
        "alpha_min",
        "alpha_max",
    ]]
    .mean()
    .reset_index()
)

summary_bap2_age = (
    df_final_bap2
    .groupby(["age", "parent_scale"])
    [[
        "S_id_child",
        "S_p_father",
        "S_m_mother",
        "S_family",
        "balance_abs_gap",
        "alpha_mean",
        "alpha_std",
        "alpha_min",
        "alpha_max",
    ]]
    .mean()
    .reset_index()
)

print("\n==============================")
print("Global summary BAP2 step4689")
print("==============================")
display(summary_bap2_global)

print("\n==============================")
print("Summary by age BAP2 step4689")
print("==============================")
display(summary_bap2_age)

summary_bap2_global.to_csv(f"{OUT_DIR}/summary_global_bap2.csv", index=False)
summary_bap2_age.to_csv(f"{OUT_DIR}/summary_age_bap2.csv", index=False)

print("\n✓ TEST FINAL BAP2 terminado")
print("CSV:", OUT_CSV)
print("Images:", IMG_DIR)
print("Summary:", OUT_DIR)

Checkpoint: /content/drive/MyDrive/Notebook4/results/ckpt/bap2_4689/bap2_s4689.pt
Test triads: 208
Edades: [5, 15, 25, 40, 60]
Parent scales: [0.0, 0.1]
Total images: 2080
OUT_DIR: /content/drive/MyDrive/Notebook4/results/eval/bap2
✓ Checkpoint loaded: step=4689, gates=16/16

Triada 1/208 | gender=woman


  0%|          | 0/30 [00:00<?, ?it/s]

age=5 | λ=0.0 | S_id=0.0340 | S_p=-0.0140 | S_m=0.0370 | S_family=0.0115 | gap=0.0510 | alpha_std=0.02621


  0%|          | 0/30 [00:00<?, ?it/s]

age=5 | λ=0.1 | S_id=0.0646 | S_p=-0.0089 | S_m=0.0155 | S_family=0.0033 | gap=0.0245 | alpha_std=0.02626


...

age=60 | λ=0.1 | S_id=0.2375 | S_p=0.0054 | S_m=0.0595 | S_family=0.0325 | gap=0.0542 | alpha_std=0.02613

Filas finales BAP2
2080
Esperadas: 2080
✓ Row count is correct.

Global summary BAP2 step4689


,parent_scale,S_id_child,S_p_father,S_m_mother,S_family,balance_abs_gap,alpha_mean,alpha_std,alpha_min,alpha_max
0,0.0,0.186944,0.011409,0.017132,0.014271,0.058377,0.498077,0.026752,0.366957,0.62997
1,0.1,0.186964,0.017099,0.023106,0.020102,0.058842,0.498077,0.026742,0.366942,0.62978



Summary by age BAP2 step4689


,age,parent_scale,S_id_child,S_p_father,S_m_mother,S_family,balance_abs_gap,alpha_mean,alpha_std,alpha_min,alpha_max
0,5,0.0,0.228909,0.015005,0.016991,0.015998,0.057917,0.498078,0.026651,0.368219,0.630570
1,5,0.1,0.229202,0.021431,0.021773,0.021602,0.057876,0.498079,0.026645,0.368137,0.630254
2,15,0.0,0.174865,0.009960,0.017790,0.013875,0.052823,0.498081,0.026513,0.368554,0.628452
3,15,0.1,0.174445,0.014473,0.021724,0.018098,0.051596,0.498081,0.026501,0.368595,0.628218
4,25,0.0,0.176835,0.006010,0.009614,0.007812,0.054142,0.498073,0.026682,0.367497,0.629344
5,25,0.1,0.175151,0.011241,0.016735,0.013988,0.055995,0.498075,0.026671,0.367485,0.629258
6,40,0.0,0.172542,0.008963,0.016993,0.012978,0.061187,0.498073,0.026901,0.365745,0.630306
7,40,0.1,0.176098,0.015348,0.024192,0.019770,0.061390,0.498074,0.026891,0.365732,0.630115
8,60,0.0,0.181572,0.017110,0.024274,0.020692,0.065817,0.498079,0.027011,0.364771,0.631180
9,60,0.1,0.179924,0.023001,0.031106,0.027053,0.067352,0.498079,0.027001,0.364758,0.631055



✓ TEST FINAL BAP2 terminado
CSV: /content/drive/MyDrive/Notebook4/results/eval/bap2/bap2.csv
Images: /content/drive/MyDrive/Notebook4/results/eval/bap2/images
Summary: /content/drive/MyDrive/Notebook4/results/eval/bap2


In [11]:
# ============================================================
# Paired statistics BAP2 TEST
# λ=0.1 vs λ=0.0
# CPU only
# ============================================================

import os
import numpy as np
import pandas as pd
from IPython.display import display

try:
    from scipy.stats import wilcoxon
    HAS_SCIPY = True
except Exception as e:
    HAS_SCIPY = False
    print("⚠ scipy not available:", e)

OUT_DIR = f"{DRIVE}/results/eval/bap2"
CSV_PATH = f"{OUT_DIR}/bap2.csv"

df = pd.read_csv(CSV_PATH)

print("Filas:", len(df))
print("Parent scales:", df["parent_scale"].unique())
print("Triads:", df["triad_idx"].nunique())
print("Ages:", sorted(df["age"].unique()))

metrics = [
    "S_id_child",
    "S_p_father",
    "S_m_mother",
    "S_family",
    "balance_abs_gap",
]

df0 = df[df["parent_scale"] == 0.0].copy()
df1 = df[df["parent_scale"] == 0.1].copy()

key_cols = ["triad_idx", "age"]

merged = df0[key_cols + metrics].merge(
    df1[key_cols + metrics],
    on=key_cols,
    suffixes=("_0", "_1")
)

print("Pares:", len(merged))

rows = []

rng = np.random.default_rng(123)
B = 5000

for m in metrics:
    d = merged[f"{m}_1"] - merged[f"{m}_0"]

    boot = []
    arr = d.values

    for _ in range(B):
        sample = rng.choice(arr, size=len(arr), replace=True)
        boot.append(sample.mean())

    boot = np.array(boot)

    row = {
        "metric": m,
        "mean_lambda0": merged[f"{m}_0"].mean(),
        "mean_lambda01": merged[f"{m}_1"].mean(),
        "mean_delta_0.1_minus_0.0": d.mean(),
        "median_delta": d.median(),
        "ci95_low": np.percentile(boot, 2.5),
        "ci95_high": np.percentile(boot, 97.5),
        "win_rate_delta_gt_0": (d > 0).mean(),
        "n_pairs": len(d),
    }

    if HAS_SCIPY:
        try:
            stat, p = wilcoxon(d)
            row["wilcoxon_stat"] = stat
            row["wilcoxon_p"] = p
        except Exception as e:
            row["wilcoxon_stat"] = None
            row["wilcoxon_p"] = None
            row["wilcoxon_error"] = str(e)

    rows.append(row)

df_bap2_stats = pd.DataFrame(rows)

print("\n==============================")
print("Paired statistics BAP2 λ=0.1 vs λ=0.0")
print("==============================")
display(df_bap2_stats)

# By age
age_rows = []

for age in sorted(merged["age"].unique()):
    sub = merged[merged["age"] == age]

    for m in metrics:
        d = sub[f"{m}_1"] - sub[f"{m}_0"]

        age_rows.append({
            "age": age,
            "metric": m,
            "mean_delta_0.1_minus_0.0": d.mean(),
            "median_delta": d.median(),
            "win_rate_delta_gt_0": (d > 0).mean(),
            "n_pairs": len(d),
        })

df_bap2_stats_age = pd.DataFrame(age_rows)

print("\n==============================")
print("Statistics by age")
print("==============================")
display(df_bap2_stats_age)

df_bap2_stats.to_csv(f"{OUT_DIR}/stats_paired_bap2_l01_vs_lambda0.csv", index=False)
df_bap2_stats_age.to_csv(f"{OUT_DIR}/stats_by_age_bap2_l01_vs_lambda0.csv", index=False)

print("\n✓ Saved to:", OUT_DIR)

Filas: 2080
Parent scales: [0.  0.1]
Triads: 208
Edades: [np.int64(5), np.int64(15), np.int64(25), np.int64(40), np.int64(60)]
Pares: 1040

Paired statistics BAP2 λ=0.1 vs λ=0.0


,metric,mean_lambda0,mean_lambda01,mean_delta_0.1_minus_0.0,median_delta,ci95_low,ci95_high,win_rate_delta_gt_0,n_pairs,wilcoxon_stat,wilcoxon_p
0,S_id_child,0.186944,0.186964,0.000020,-0.000127,-0.001574,0.001608,0.492308,1040,269627.0,9.150926e-01
1,S_p_father,0.011409,0.017099,0.005689,0.005642,0.004543,0.006843,0.693269,1040,153045.0,6.545863e-34
2,S_m_mother,0.017132,0.023106,0.005974,0.006525,0.004805,0.007138,0.700962,1040,149108.0,4.204924e-36
3,S_family,0.014271,0.020102,0.005831,0.005978,0.004973,0.006686,0.719231,1040,121298.0,1.280611e-53
4,balance_abs_gap,0.058377,0.058842,0.000465,-0.000064,-0.000925,0.001863,0.498077,1040,268390.0,8.147600e-01



Statistics by age


,age,metric,mean_delta_0.1_minus_0.0,median_delta,win_rate_delta_gt_0,n_pairs
0,5,S_id_child,0.000293,0.000706,0.514423,208
1,5,S_p_father,0.006426,0.005774,0.764423,208
2,5,S_m_mother,0.004782,0.006381,0.668269,208
3,5,S_family,0.005604,0.005964,0.740385,208
4,5,balance_abs_gap,-0.000042,-0.000063,0.500000,208
5,15,S_id_child,-0.000420,-0.000524,0.480769,208
6,15,S_p_father,0.004513,0.004859,0.653846,208
7,15,S_m_mother,0.003934,0.004793,0.663462,208
8,15,S_family,0.004224,0.003695,0.649038,208
9,15,balance_abs_gap,-0.001226,-0.000766,0.475962,208



✓ Saved to: /content/drive/MyDrive/Notebook4/results/eval/bap2


In [12]:
# ============================================================
# Check the NoGate implementation available in src
# Does not modify the pipeline. Only inspects imports and names.
# ============================================================

import os, sys, importlib, inspect

candidate_modules = [
    "src.bap_processor_no_gate",
    "src.attach_bap",
    "src.attach_child_anchor_bap_gate",
    "src.bap_child_anchor_gate",
    "src.bap2",
    "src.att2",
]

for modname in candidate_modules:
    print("\n" + "="*80)
    print("MODULE:", modname)
    try:
        mod = importlib.import_module(modname)
        print("✓ Importa")

        names = [
            n for n in dir(mod)
            if (
                "Gate" in n
                or "gate" in n
                or "BAP" in n
                or "bap" in n
                or "attach" in n
                or "No" in n
                or "no" in n
                or "Child" in n
                or "Anchor" in n
            )
        ]

        print("Nombres candidatos:")
        for n in names:
            obj = getattr(mod, n)
            print(" -", n, "|", type(obj))

        print("\nFunctions/classes with signature:")
        for n in names:
            obj = getattr(mod, n)
            if inspect.isfunction(obj) or inspect.isclass(obj):
                try:
                    print(f"\n{n}:")
                    print(inspect.signature(obj))
                except Exception as e:
                    print(f"{n}: no signature ({e})")

    except Exception as e:
        print("✗ Import failed")
        print(type(e).__name__, e)


MODULE: src.bap_processor_no_gate
✓ Importa
Nombres candidatos:

Functions/classes with signature:

MODULE: src.attach_bap
✗ No importa
ModuleNotFoundError No module named 'src.attach_bap'

MODULE: src.attach_child_anchor_bap_gate
✓ Importa
Nombres candidatos:
 - ChildAnchorBAPGateAttnProcessor | <class 'type'>
 - attach_child_anchor_bap_gate_to_unet | <class 'function'>

Functions/classes with signature:

ChildAnchorBAPGateAttnProcessor:
(hidden_size: int, cross_attention_dim: int, num_tokens: int = 4, scale: float = 1.0, child_scale: float = 1.0, parent_scale: float = 0.1, gate_groups: int = 16)

attach_child_anchor_bap_gate_to_unet:
(unet, num_tokens: int = 4, scale: float = 1.0, child_scale: float = 1.0, parent_scale: float = 0.1, gate_groups: int = 16, device='cuda', dtype=torch.float16, copy_ip_weights: bool = True)

MODULE: src.bap_child_anchor_gate
✓ Importa
Nombres candidatos:
 - ChildAnchorBAPGateAttnProcessor | <class 'type'>

Functions/classes with signature:

ChildAnchorB

In [ ]:
# ============================================================
# TEST FINAL — NoGate / FixedGate α=0.5
# 208 triads × 5 ages × λ=0.1 = 1040 images
#
# Baseline:
#   child anchor + father + mother
#   but WITHOUT a learned gate
#   alpha = 0.5 fijo
# ============================================================

import os
import torch
import torch.nn.functional as F
import pandas as pd
from torch.utils.data._utils.collate import default_collate
from IPython.display import display

# ------------------------------------------------------------
# Verificaciones
# ------------------------------------------------------------

required = [
    "test_ds",
    "bap2_modules",
    "generate_bap2_final",
    "arcface_embed_from_pil",
    "ensure_2d",
]

missing = [x for x in required if x not in globals()]
assert len(missing) == 0, f"Missing objects in memory: {missing}"

assert len(bap2_modules) == 16, "bap2_modules must have 16 modules."

# ------------------------------------------------------------
# Configuration
# ------------------------------------------------------------

AGES = [5, 15, 25, 40, 60]
PARENT_SCALE = 0.1

BASE_SEED = 42

OUT_DIR = f"{DRIVE}/results/eval/nogate"
IMG_DIR = f"{OUT_DIR}/images"
os.makedirs(OUT_DIR, exist_ok=True)
os.makedirs(IMG_DIR, exist_ok=True)

OUT_CSV = f"{OUT_DIR}/nogate.csv"

print("Test triads:", len(test_ds))
print("Ages:", AGES)
print("Parent scale:", PARENT_SCALE)
print("Total images:", len(test_ds) * len(AGES))
print("OUT_DIR:", OUT_DIR)

# ------------------------------------------------------------
# Forzar NoGate: alpha = sigmoid(0) = 0.5
# ------------------------------------------------------------

for name, proc in bap2_modules.items():
    proc.gate.float()
    with torch.no_grad():
        proc.gate.weight.zero_()
        proc.gate.bias.zero_()
    proc.gate.to(DEVICE)

print("✓ Gates reseteados a cero: alpha fijo = 0.5")

# Sanity check alpha
with torch.no_grad():
    first_name, first_proc = next(iter(bap2_modules.items()))
    dummy = torch.randn(
        1,
        first_proc.gate.in_features,
        device=DEVICE,
        dtype=torch.float32,
    )
    alpha = torch.sigmoid(first_proc.gate(dummy))

print("Alpha sanity:")
print("mean:", alpha.mean().item())
print("std :", alpha.std().item())
print("min :", alpha.min().item())
print("max :", alpha.max().item())

assert abs(alpha.mean().item() - 0.5) < 1e-6
assert alpha.std().item() < 1e-6

# ------------------------------------------------------------
# Saving NoGate images
# ------------------------------------------------------------

def save_nogate_image(img, triad_idx, age):
    subdir = f"{IMG_DIR}/age_{age}"
    os.makedirs(subdir, exist_ok=True)

    path = f"{subdir}/t{triad_idx:04d}_a{int(age):02d}.jpg"
    img.save(path, quality=95)
    return path

# ------------------------------------------------------------
# Automatic resume
# ------------------------------------------------------------

rows = []
done = set()

if os.path.exists(OUT_CSV):
    old = pd.read_csv(OUT_CSV)
    rows = old.to_dict("records")
    done = set(zip(old["triad_idx"], old["age"]))
    print(f"✓ Resume detected: {len(old)} existing rows")

# ------------------------------------------------------------
# Loop principal
# ------------------------------------------------------------

for triad_idx in range(len(test_ds)):

    sample = test_ds[triad_idx]

    batch = default_collate([sample])
    batch = {
        k: v.to(DEVICE) if isinstance(v, torch.Tensor) else v
        for k, v in batch.items()
    }

    h_c = ensure_2d(batch["h_c"])
    h_p = ensure_2d(batch["h_p"])
    h_m = ensure_2d(batch["h_m"])

    h_c_eval = F.normalize(h_c.float(), p=2, dim=1)
    h_p_eval = F.normalize(h_p.float(), p=2, dim=1)
    h_m_eval = F.normalize(h_m.float(), p=2, dim=1)

    gender = sample.get("gender", "person")
    if gender not in ["man", "woman", "person"]:
        gender = "person"

    print(f"\nNoGate | Triada {triad_idx + 1}/{len(test_ds)} | gender={gender}")

    for age in AGES:

        if (triad_idx, age) in done:
            print(f"skip triad={triad_idx} age={age}")
            continue

        paired_seed = BASE_SEED + triad_idx * 1000 + age

        img, alpha_stats, prompt = generate_bap2_final(
            h_c=h_c,
            h_p=h_p,
            h_m=h_m,
            gender=gender,
            age=age,
            parent_scale=PARENT_SCALE,
            seed=paired_seed,
        )

        img_path = save_nogate_image(
            img=img,
            triad_idx=triad_idx,
            age=age,
        )

        h_gen = arcface_embed_from_pil(img)

        S_id = F.cosine_similarity(h_gen, h_c_eval).item()
        S_p  = F.cosine_similarity(h_gen, h_p_eval).item()
        S_m  = F.cosine_similarity(h_gen, h_m_eval).item()

        row = {
            "variant": "NoGate_fixed_alpha05",
            "checkpoint_step": "fixed_alpha_0.5",
            "triad_idx": triad_idx,
            "age": age,
            "parent_scale": PARENT_SCALE,
            "gender": gender,
            "S_id_child": S_id,
            "S_p_father": S_p,
            "S_m_mother": S_m,
            "S_family": 0.5 * (S_p + S_m),
            "balance_abs_gap": abs(S_p - S_m),
            "alpha_mean": alpha_stats["alpha_mean"],
            "alpha_std": alpha_stats["alpha_std"],
            "alpha_min": alpha_stats["alpha_min"],
            "alpha_max": alpha_stats["alpha_max"],
            "key_p": sample.get("key_p", ""),
            "key_m": sample.get("key_m", ""),
            "key_c": sample.get("key_c", ""),
            "seed": paired_seed,
            "prompt": prompt,
            "image_path": img_path,
        }

        rows.append(row)

        # Incremental saving
        pd.DataFrame(rows).to_csv(OUT_CSV, index=False)

        print(
            f"age={age} | "
            f"S_id={S_id:.4f} | "
            f"S_p={S_p:.4f} | "
            f"S_m={S_m:.4f} | "
            f"S_family={row['S_family']:.4f} | "
            f"gap={row['balance_abs_gap']:.4f} | "
            f"alpha_mean={alpha_stats['alpha_mean']:.4f}"
        )

# ------------------------------------------------------------
# Final summary NoGate
# ------------------------------------------------------------

df_final_nogate = pd.DataFrame(rows)

print("\n==============================")
print("Filas finales NoGate")
print("==============================")
print(len(df_final_nogate))

expected_rows = len(test_ds) * len(AGES)
print("Esperadas:", expected_rows)

if len(df_final_nogate) != expected_rows:
    print("⚠ Warning: unexpected number of rows. Check whether there was an interruption.")
else:
    print("✓ Row count is correct.")

summary_nogate_global = (
    df_final_nogate
    .groupby("variant")
    [[
        "S_id_child",
        "S_p_father",
        "S_m_mother",
        "S_family",
        "balance_abs_gap",
        "alpha_mean",
        "alpha_std",
        "alpha_min",
        "alpha_max",
    ]]
    .mean()
    .reset_index()
)

summary_nogate_age = (
    df_final_nogate
    .groupby(["age"])
    [[
        "S_id_child",
        "S_p_father",
        "S_m_mother",
        "S_family",
        "balance_abs_gap",
        "alpha_mean",
        "alpha_std",
        "alpha_min",
        "alpha_max",
    ]]
    .mean()
    .reset_index()
)

print("\n==============================")
print("Global summary NoGate")
print("==============================")
display(summary_nogate_global)

print("\n==============================")
print("Summary by age NoGate")
print("==============================")
display(summary_nogate_age)

summary_nogate_global.to_csv(f"{OUT_DIR}/summary_global_nogate.csv", index=False)
summary_nogate_age.to_csv(f"{OUT_DIR}/summary_age_nogate.csv", index=False)

print("\n✓ TEST FINAL NoGate terminado")
print("CSV:", OUT_CSV)
print("Images:", IMG_DIR)
print("Summary:", OUT_DIR)

Test triads: 208
Edades: [5, 15, 25, 40, 60]
Parent scale: 0.1
Total images: 1040
OUT_DIR: /content/drive/MyDrive/Notebook4/results/eval/nogate
✓ Gates reseteados a cero: alpha fijo = 0.5
Alpha sanity:
mean: 0.5
std : 0.0
min : 0.5
max : 0.5

NoGate | Triada 1/208 | gender=woman


  0%|          | 0/30 [00:00<?, ?it/s]

age=5 | S_id=0.0631 | S_p=-0.0072 | S_m=0.0150 | S_family=0.0039 | gap=0.0222 | alpha_mean=0.5000


  0%|          | 0/30 [00:00<?, ?it/s]

age=15 | S_id=0.0610 | S_p=-0.0061 | S_m=-0.0389 | S_family=-0.0225 | gap=0.0329 | alpha_mean=0.5000


...

  0%|          | 0/30 [00:00<?, ?it/s]

age=5 | S_id=0.2870 | S_p=0.0452 | S_m=0.0598 | S_family=0.0525 | gap=0.0146 | alpha_mean=0.5000


  0%|          | 0/30 [00:00<?, ?it/s]

age=15 | S_id=0.2272 | S_p=0.1279 | S_m=0.0133 | S_family=0.0706 | gap=0.1147 | alpha_mean=0.5000


  0%|          | 0/30 [00:00<?, ?it/s]

In [ ]:
sd = ckpt["bap2_gate_state"]

first_key = list(sd.keys())[0]
print("Primera capa:", first_key)
print(type(sd[first_key]))

if isinstance(sd[first_key], dict):
    print("Keys inside one layer:")
    for k, v in sd[first_key].items():
        if hasattr(v, "shape"):
            print(k, v.shape, v.dtype)
        else:
            print(k, type(v), v)
else:
    print(sd[first_key])

Primera capa: down_blocks_0_attentions_0_transformer_blocks_0_attn2
<class 'collections.OrderedDict'>
Keys inside one layer:
weight torch.Size([16, 2176]) torch.float32
bias torch.Size([16]) torch.float32
